<a href="https://colab.research.google.com/github/jwadv/uni-public/blob/main/CR/geffe_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Geffe Generator

##Indice
1. Esercizio 4.18
2. Implementazione del Cifrario (Cifratura e Decifratura)
3. Attacco di Correlazione a chiaro Noto

Riferimenti al testo: pp. 36-38, 122


---

## 1. Esercizio 4.18


### Testo
Il **Generatore di Geffe** è una funzione combinante $F : (\mathbb{Z}_2)^3 \to \mathbb{Z}_2$ definita dalla seguente formula booleana:

$$F(z_1, z_2, z_3) = (z_1 \land z_2) \oplus (\neg z_1 \land z_3)$$

Determinare le correlazioni tra i singoli ingressi $z_1, z_2, z_3$ e l'uscita $z = F(z_1, z_2, z_3)$, seguendo il procedimento mostrato nella Sezione 4.8.1 per la funzione di maggioranza.

### Svolgimento

Si ipotizza che le variabili di ingresso $(z_1, z_2, z_3)$ siano indipendenti e distribuite in modo uniforme, rendendo le $2^3 = 8$ possibili terne equiprobabili (ciascuna con probabilità $\frac{1}{8}$).

Fatta questa premessa, si stilano le tabelle di verità a partire dalle quali è possibile verificare per quali righe l'uscita $z = F(z_1, z_2, z_3)$ coincide con i singoli ingressi $z_1, z_2, z_3$.

| $z_1$ | $z_2$ | $z_3$ | $F(z_1, z_2, z_3)$ |
| :---: | :---: | :---: | :----------------: |
|  $0$  |  $0$  |  $0$  |      **$0$**       |
|  $0$  |  $0$  |  $1$  |      **$1$**       |
|  $0$  |  $1$  |  $0$  |      **$0$**       |
|  $0$  |  $1$  |  $1$  |      **$1$**       |
|  $1$  |  $0$  |  $0$  |      **$0$**       |
|  $1$  |  $0$  |  $1$  |      **$0$**       |
|  $1$  |  $1$  |  $0$  |      **$1$**       |
|  $1$  |  $1$  |  $1$  |      **$1$**       |

Associate le variabili di ingresso e di uscita alle rispettive variabili aleatorie $\mathbf{z_1}, \mathbf{z_2}, \mathbf{z_3}$ e $\mathbf{z}$, si procede con il calcolo delle correlazioni tra i singoli ingressi e l'uscita della funzione combinante (basta contare quante volte $z = F(z_1, z_2, z_3)$ per ciascun ingresso).

* $\mathbf{z_1}$: $$\Pr[\mathbf{z} = \mathbf{z_1}] = \frac{4}{8} = \frac{1}{2} = 0.5$$
* $\mathbf{z_2}$: $$\Pr[\mathbf{z} = \mathbf{z_2}] = \frac{6}{8} = \frac{3}{4} = 0.75$$
* $\mathbf{z_3}$: $$\Pr[\mathbf{z} = \mathbf{z_3}] = \frac{6}{8} = \frac{3}{4} = 0.75$$

Dai calcoli ottenuti si evince che:
* essendo la probabilità di accordo tra l'uscita e l'ingresso $z_1$ del 50%, non vi è correlazione tra questi due valori
* c'è una forte correlazione (75%) tra l'uscita e l'ingresso $z_2$: quando il selettore $z_1 = 1$, la funzione equivale esattamente a $F = z_2$
* c'è una forte correlazione (75%) tra l'uscita e l'ingresso $z_3$: quando il selettore $z_0 = 1$, la funzione equivale esattamente a $F = z_3$

Come si osserverà nella simulazione dell'attacco per correlazione, le forti correlazioni del 75% compromettono la sicurezza del cifrario e costituiscono il punto debole del Generatore di Geffe.

---

## 2. Implementazione del Cifrario (Cifratura e Decifratura)

Il programma che segue permette di cifrare e decifrare con un generatore di Geffe e di attaccarlo.

Definire gli LFSRs (linear feedback shift register) implementati.

Per prima cosa si definiscono i parametri.
`t1`. `t2`, e `t3` sono i "punti di presa: le posizioni specifiche all'interno del registro da cui vengono estratti i bit per calcolare il bit di retroazione tramite l'operazione logica XOR.

In questa implementazione sono state scelte tap corrispondenti a polinomi primitivi:
1. $P_1(x) = x^4 + x + 1$
2. $P_2(x) = x^5 + x^2 + 1$
3. $P_3(x) = x^6 + x + 1$

* `t1 = [0, 1]` (4 bit): nuovo bit = (valore cella 0) $\oplus$ (valore cella 1)
* `t2 = [0, 2]` (5 bit): nuovo bit = (valore cella 0) $\oplus$ (valore cella 2)
* `t3 = [0, 1]` (6 bit): nuovo bit = (valore cella 0) $\oplus$ (valore cella 1)

La scelta implementativa garantisce un periodo di ripetizione della sequenza generata con il tempo massimo possibile ($2^n - 1$ passi, dove $n$ è il numero di bit del registro).

La decisione di usare 4, 5 e 6 bit diversi per ciascun registro è puramente dettato dal desiderio di irrobustire la sicurezza del cifrario.

In [ ]:
import random

t1 = [0,1]
t2 = [0,2]
t3 = [0,1]


### 2.2 Decifratura

La logica della decifratura può essere riassunta in questa espressione: $$\boxed{\text{plaintext = ciphertext ⊕ keystream}}$$

* input: testo cifrato esadecimale
* output: testo in chiaro
* idea: usa lo stesso keystream e applica XOR di nuovo

Ovviamente devono essere noti s1, s2 e s3.

Il testo cifrato richiesto in hex viene convertito in bit.
Poi, una volta ricostruito la keystream tramite la funzione `geffe`, viene eseguito lo XOR tra il cifrato in bit e la keystream.

---

## 3. Attacco di Correlazione a chiaro noto

L'attacco di Correlazione è una stratega di attacco valida nel caso in cui esistono correlazioni tra gli output delle LFSRs (che sono gli input della funzone combinante) e quello della funzione combinante.

A seguire si presenta una simulazione d'attacco di Correlazione a chiaro noto sul cifrario a flussi con il Geffe Generator dell'Esercizio di riferimento.

* Il messaggio del mittente viene cifrato tramite la funzione combinante della Geffe Generator.
* L'attaccante conosce solo un pezzo iniziale del messaggio e a partire da esso tramite il cipher text intercettato è in grado di risalire a una parte del keystream perché $$\boxed{\text{plaintext = ciphertext ⊕ keystream}} \implies \boxed{\text{keystream = ciphertext ⊕ plaintext}}$$

Gli stati iniziali dei tre registri vengono cercati uno alla volta seguendo un approccio divide et impera (devide and conquer) che sfrutta la debollezza della funzione combinante.

Il risultato è una stima, non una certezza. Nel generatore Geffe, l’uscita è correlata soprattutto alle sequenze dei registri 2 e 3: ci si aspetta una corrispondenza vicina al 75% con il registro corretto, mentre il registro 1 non è recuperabile con questo semplice confronto diretto. Con un tratto di keystream breve, inoltre, un candidato sbagliato può ottenere per caso un punteggio alto.